In [ ]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("practiceSet2").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())


In [ ]:
!pip install pyspark

In [ ]:
shipments_df = spark.read.csv(
    "shipments_75000.csv",
    header=True,
    inferSchema=True
)

PART A | DataFrame Operations

01 Display the schema, total number of records, and first 15 shipments.

02 Select shipment_id, warehouse_id, service_level, shipment_status, and shipping_fee_inr.

03 Filter shipments where service_level = "Express" and package_weight_kg > 5.

04 Sort shipments by shipping_fee_inr in descending order and display the top 20





In [ ]:
#1
shipments_df.printSchema()
print("Total records:", shipments_df.count())
shipments_df.show(15)


In [ ]:
#2
shipments_df.select("shipment_id","warehouse_id","service_level","shipment_status","shipping_fee_inr").show(5)

In [ ]:
#3
from pyspark.sql.functions import col,min,max,sum,avg
shipments_df.filter(col("service_level")=="Express").show(5)

In [ ]:
#4
shipments_df.orderBy(col("shipping_fee_inr").desc()).show(20)

PART B | Aggregations and Joins

05 Calculate the total number of shipments for each shipment_status.

06 Calculate total shipping fees, total shipping costs, and profit margin for each service_level.Handle missing cost values appropriately.

07 Join shipments with the warehouse master and calculate the number of shipments originating
from each region

08 Join shipments, carriers, and routes. Calculate average shipping cost per kilometre for each
carrier, excluding missing costs and zero distances.



In [ ]:
#5
shipments_df.groupBy(col("shipment_status")).count().show()

In [ ]:
#6
shipments_df = shipments_df.filter(col("shipping_cost_inr").isNotNull())

df = (shipments_df
      .groupBy(col("service_level"))
      .agg(
          sum("shipping_fee_inr").alias("total_shipping_fee"),
          sum("shipping_cost_inr").alias("total_shipping_cost")
      )
      .withColumn("profit", col("total_shipping_fee") - col("total_shipping_cost"))
      .withColumn("profit_margin_pct", (col("profit") / col("total_shipping_fee")) * 100)
)

df.show()

In [ ]:
warehouse_df=spark.read.csv("warehouse_master.csv",header=True,inferSchema=True)
warehouse_df.show(3)

In [ ]:
#7
join_df=shipments_df.join(warehouse_df,on="warehouse_id",how="inner")
join_df.groupBy(col("region")).count().show()

In [ ]:
carriers_df=spark.read.csv("carrier_master.csv",header=True,inferSchema=True)
carriers_df.show(3)

In [ ]:
routes_df=spark.read.csv("route_master.csv",header=True,inferSchema=True)
routes_df.show(3)

In [ ]:
#8
join_df=(shipments_df.join(routes_df,on="route_id",how="inner")
.join(carriers_df,on="carrier_id",how="inner")
)
df=(join_df.filter(col("shipping_cost_inr").isNotNull() & (col("distance_km") > 0))
    .withColumn("cost_per_km",col("shipping_cost_inr")/col("distance_km"))
    .groupBy("carrier_id")
    .agg(avg("cost_per_km").alias("average_shipping_cost_per_km"))
    )
df.show()

PART C | Null Handling and Conditional Logic

09 Count missing values in carrier_id, package_weight_kg, shipping_cost_inr, and customer_rating.

10 Replace null handling_minutes values with zero in one DataFrame. Create another DataFrame that excludes shipments with a missing carrier_id.

11 Create weight_category using when() and otherwise(): Light (< 2 kg), Medium (2-10 kg), Heavy
(> 10 kg), and Unknown (null).



In [ ]:
#9
from pyspark.sql.functions import when
shipments_df.select(
    sum(when(col("carrier_id").isNull(), 1).otherwise(0)).alias("missing_carrier_id"),
    sum(when(col("package_weight_kg").isNull(), 1).otherwise(0)).alias("missing_package_weight"),
    sum(when(col("shipping_cost_inr").isNull(), 1).otherwise(0)).alias("missing_shipping_cost"),
    sum(when(col("customer_rating").isNull(), 1).otherwise(0)).alias("missing_customer_rating")
).show()

In [ ]:
#10
shipments_df=shipments_df.na.fill({"handling_minutes":0})
join_df.filter(col("carrier_id").isNotNull()).show()

In [ ]:
#11
df = join_df.withColumn(
    "weight_category",
    when(col("package_weight_kg").isNull(), "Unknown")
    .when(col("package_weight_kg") < 2, "Light")
    .when(col("package_weight_kg") <= 10, "Medium")
    .otherwise("Heavy")
)

df.show(3)

PART D | Date and Time Functions

12 Convert delivery-related date columns to Spark date types and calculate delivery duration in days for delivered shipments.

13 Create delivery_performance as Early, On Time, or Late by comparing actual and promised delivery dates. Give missing actual dates a separate classification.

14 Using booking_date, calculate monthly delivered-shipment counts and total shipping fees.


In [ ]:
shipments_df.show(3)

In [ ]:
#12
from pyspark.sql.functions import to_date,datediff
df = (shipments_df
      .withColumn("booking_date", to_date(col("booking_date")))
      .withColumn("dispatch_date", to_date(col("dispatch_date")))
      .withColumn("promised_delivery_date", to_date(col("promised_delivery_date")))
      .withColumn("actual_delivery_date", to_date(col("actual_delivery_date")))
      .withColumn(
          "delivery_duration_days",
          datediff(col("actual_delivery_date"), col("promised_delivery_date"))
      )
)
df.show(2)

In [ ]:
#13
df = df.withColumn(
    "delivery_performance",
    when(col("actual_delivery_date").isNull(), "Missing")
    .when(col("actual_delivery_date") < col("promised_delivery_date"), "Early")
    .when(col("actual_delivery_date") == col("promised_delivery_date"), "On Time")
    .otherwise("Late")
)

df.show(3)

In [ ]:
#14
from pyspark.sql.functions import col, month, year, sum, count

df = (shipments_df
      .filter(col("shipment_status") == "Delivered")
      .groupBy(year("booking_date").alias("year"),
               month("booking_date").alias("month"))
      .agg(
          count("*").alias("total_shipments"),
          sum("shipping_fee_inr").alias("total_shipping_fees")
      )
)

df.show(5)

PART E | Window Functions

15 Rank carriers by total shipping fees within each warehouse region using dense_rank().

16 Identify the latest booked shipment for each warehouse using row_number().

17 Calculate a running total of shipping fees for each warehouse, ordered by booking date and shipment ID

In [ ]:
from pyspark.sql.functions import col, sum, dense_rank
from pyspark.sql.window import Window
join_df = (shipments_df.join(warehouse_df, on="warehouse_id", how="inner")
                      .join(carriers_df, on="carrier_id", how="inner"))

df = (join_df
      .groupBy("region", "carrier_id")
      .agg(sum("shipping_fee_inr").alias("total_shipping_fees"))
)

window_spec = Window.partitionBy("region").orderBy(col("total_shipping_fees").desc())

df = df.withColumn("rank", dense_rank().over(window_spec))

df.show()

In [ ]:
#16
from pyspark.sql.functions import row_number


window_spec = Window.partitionBy("warehouse_id").orderBy(col("booking_date").desc())

df = (shipments_df
      .withColumn("row_num", row_number().over(window_spec))
      .filter(col("row_num") == 1)
)

df.show()

In [ ]:
#17
window_spec = (Window
               .partitionBy("warehouse_id")
               .orderBy("booking_date", "shipment_id")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))

df = shipments_df.withColumn(
    "running_total",
    sum("shipping_fee_inr").over(window_spec)
)

df.show(5)

PART F | Union and Complex JSON

18 Combine shipments_df and extra_df using unionByName(). Count the combined records.

19 Identify repeated shipment_id values and remove duplicate shipment records. Compare row counts before and after deduplication.

20 From the nested sensor JSON, extract warehouse ID, temperature limits, and sensor details. Use explode() to produce one row per sensor




In [ ]:
extra_df=spark.read.csv("shipments_extra_12000.csv",header=True,inferSchema=True)

In [ ]:
#18
combined_df = shipments_df.unionByName(extra_df)

print(combined_df.count())

In [ ]:
#19
combined_df.groupBy("shipment_id").count() \
    .filter(col("count") > 1).show()

df = combined_df.dropDuplicates(["shipment_id"])

print("Before:", combined_df.count())
print("After:", df.count())

In [ ]:
sensor_df = spark.read.option("multiline", "true").json("warehouse_sensor_config.json")

sensor_df.printSchema()
sensor_df.show(truncate=False)

In [ ]:
#20
from pyspark.sql.functions import col, explode


df = sensor_df.select(
    col("warehouse_id"),
    col("environment.temperature_range_c.minimum").alias("min_temperature"),
    col("environment.temperature_range_c.maximum").alias("max_temperature"),
    explode(col("sensors")).alias("sensor")
)

df = df.select(
    col("warehouse_id"),
    col("min_temperature"),
    col("max_temperature"),
    col("sensor.sensor_id").alias("sensor_id"),
    col("sensor.type").alias("sensor_type"),
    col("sensor.enabled").alias("enabled")
)

df.show(truncate=False)

PART G | Lazy Execution, Caching, and Persistence

21 Build a pipeline using filter(), join(), withColumn(), groupBy(), and orderBy(). Inspect its plan with
explain() before calling an action.

22 Cache delivered shipments. Compare execution times for repeated count() and aggregation actions.

23 Persist Express shipments using MEMORY_AND_DISK. Run two actions, inspect the storage level,and release the data using unpersist().

In [ ]:
#21
df = shipments_df.filter(col("shipping_cost_inr") > 500)

df = df.withColumn(
    "cost_with_tax",
    col("shipping_cost_inr") * 1.18
)

df.explain()

In [ ]:
#22
from pyspark.sql.functions import col
import time

# Filter delivered shipments and cache the result
delivered_df = shipments_df.filter(
    col("shipment_status") == "Delivered"
).cache()

# First action: count
start = time.time()
print("Total delivered shipments:", delivered_df.count())
print("Count execution time:", time.time() - start)

# Second action: aggregation
start = time.time()
delivered_df.groupBy("service_level").count().show()
print("Aggregation execution time:", time.time() - start)

# Release cached data
delivered_df.unpersist()

In [ ]:
#23
from pyspark import StorageLevel
from pyspark.sql.functions import col

# Filter Express shipments and persist
express_df = shipments_df.filter(
    col("service_level") == "Express"
).persist(StorageLevel.MEMORY_AND_DISK)

# Action 1
print("Total Express shipments:", express_df.count())

# Action 2
express_df.groupBy("shipment_status").count().show()

# Check storage level
print("Storage level:", express_df.storageLevel)

# Release persisted data
express_df.unpersist()

In [ ]:
#24
from pyspark.sql.functions import spark_partition_id, col

df = shipments_df.repartition(8)

df = df.withColumn("partition_id", spark_partition_id())

df.groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
#25
# Repartition into 12 partitions
df12 = shipments_df.repartition(12)

# Reduce partitions to 3 using coalesce()
df_coalesce = df12.coalesce(3)

# Reduce partitions to 3 using repartition()
df_repartition = df12.repartition(3)

# Compare execution plans
print("Coalesce execution plan:")
df_coalesce.explain()

print("Repartition execution plan:")
df_repartition.explain()

# Check partition counts
print("Coalesce partitions:", df_coalesce.rdd.getNumPartitions())
print("Repartition partitions:", df_repartition.rdd.getNumPartitions())

In [ ]:
#26
# Join shipments with warehouse master
join_df = shipments_df.join(
    warehouse_df,
    on="warehouse_id",
    how="inner"
)

# Create 4 output partitions and write as Parquet
output_df = join_df.repartition(4)

output_df.write.mode("overwrite").parquet(
    "/content/shipment_warehouse_parquet"
)

# Check the output
print("Output partitions:", output_df.rdd.getNumPartitions())

# Read the saved Parquet data
saved_df = spark.read.parquet(
    "/content/shipment_warehouse_parquet"
)

saved_df.show(5)

FINAL CHALLENGE | Logistics Performance Pipeline
Build one end-to-end PySpark pipeline that joins shipment, warehouse, carrier, and route data.
Complete all requirements below:
1. Clean missing shipping costs and package weights using a clearly explained strategy.
2. Calculate delivery duration, late-delivery indicators, and shipment profit.
3. Aggregate revenue, profit, and late-delivery percentage by region and carrier.
4. Rank carriers within each region by on-time delivery percentage.
5. Cache the final aggregated DataFrame and reuse it for multiple actions.
6. Save the final report as partitioned Parquet files.

In [ ]:
from pyspark.sql.functions import (
    col, lit, coalesce, when, to_date, datediff,
    sum, avg, count, dense_rank
)

# ==================================================
# 1. CLEAN MISSING SHIPPING COSTS AND PACKAGE WEIGHTS
# Strategy: Replace missing values with the median.
# If an entire column is null, use 0 as a fallback.
# ==================================================

cost_values = shipments_df.approxQuantile(
    "shipping_cost_inr", [0.5], 0.01
)
weight_values = shipments_df.approxQuantile(
    "package_weight_kg", [0.5], 0.01
)

median_cost = cost_values[0] if cost_values else 0
median_weight = weight_values[0] if weight_values else 0

clean_df = (
    shipments_df
    .withColumn(
        "shipping_cost_inr",
        coalesce(col("shipping_cost_inr"), lit(median_cost))
    )
    .withColumn(
        "package_weight_kg",
        coalesce(col("package_weight_kg"), lit(median_weight))
    )
)

# ==================================================
# 2. JOIN ALL DATA AND CALCULATE DELIVERY METRICS
# ==================================================

joined_df = (
    clean_df
    .join(warehouse_df, on="warehouse_id", how="inner")
    .join(carriers_df, on="carrier_id", how="inner")
    .join(routes_df, on="route_id", how="inner")
)

df = (
    joined_df
    .withColumn("dispatch_date", to_date(col("dispatch_date")))
    .withColumn(
        "actual_delivery_date",
        to_date(col("actual_delivery_date"))
    )
    .withColumn(
        "promised_delivery_date",
        to_date(col("promised_delivery_date"))
    )
    .withColumn(
        "delivery_duration_days",
        datediff(
            col("actual_delivery_date"),
            col("dispatch_date")
        )
    )
    .withColumn(
        "late_indicator",
        when(
            col("actual_delivery_date").isNull()
            | col("promised_delivery_date").isNull(),
            None
        )
        .when(
            col("actual_delivery_date") >
            col("promised_delivery_date"),
            1
        )
        .otherwise(0)
    )
    .withColumn(
        "on_time_indicator",
        when(
            col("actual_delivery_date").isNull()
            | col("promised_delivery_date").isNull(),
            None
        )
        .when(
            col("actual_delivery_date") <=
            col("promised_delivery_date"),
            1
        )
        .otherwise(0)
    )
    .withColumn(
        "profit",
        col("shipping_fee_inr") - col("shipping_cost_inr")
    )
)

# ==================================================
# 3. AGGREGATE REVENUE, PROFIT AND LATE DELIVERY %
#    BY REGION AND CARRIER
# ==================================================

report_df = (
    df
    .groupBy("region", "carrier_id")
    .agg(
        sum("shipping_fee_inr").alias("total_revenue"),
        sum("profit").alias("total_profit"),
        (avg("late_indicator") * 100).alias(
            "late_delivery_pct"
        ),
        (avg("on_time_indicator") * 100).alias(
            "on_time_delivery_pct"
        ),
        count("*").alias("total_shipments")
    )
)

# ==================================================
# 4. RANK CARRIERS WITHIN EACH REGION
#    HIGHEST ON-TIME DELIVERY % GETS RANK 1
# ==================================================

window_spec = (
    Window
    .partitionBy("region")
    .orderBy(col("on_time_delivery_pct").desc())
)

ranked_df = report_df.withColumn(
    "carrier_rank",
    dense_rank().over(window_spec)
)

# ==================================================
# 5. CACHE FINAL REPORT AND REUSE IT FOR MULTIPLE ACTIONS
# ==================================================

cached_df = ranked_df.cache()

print("Total report records:", cached_df.count())

cached_df.show(truncate=False)

cached_df.orderBy(
    col("total_profit").desc()
).show(truncate=False)

# ==================================================
# 6. SAVE FINAL REPORT AS PARTITIONED PARQUET FILES
# ==================================================

output_path = "/content/logistics_performance_report"

cached_df.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(output_path)

print("Final report saved successfully!")

# Read and verify the saved report
saved_df = spark.read.parquet(output_path)

saved_df.show(truncate=False)

# Release cached data after all actions are complete
cached_df.unpersist()